# Semantic Kernel

**Domain:** Agentic AI  ·  **runnable:** yes

A refresher on **Semantic Kernel (SK)** — Microsoft's open-source SDK for wiring LLMs into real applications. The core idea is the **Kernel**: a lightweight dependency-injection container that holds your **AI services** (model connectors) and your **plugins** (collections of callable functions), then lets you either invoke functions directly *or* let the model invoke them itself via **automatic function calling**.

Where [[crewai]] thinks in *roles and tasks* and [[langgraph]] thinks in *graphs and state*, SK thinks like an **application framework**: register capabilities with a container, mix code-functions and prompt-functions freely, and bolt on cross-cutting concerns (logging, security, caching) with **filters**. It's the framework you reach for when the LLM is one component inside a larger (often .NET/Python/Java enterprise) app.

## 1. What & Why

**Semantic Kernel** is an SDK from Microsoft for building AI applications and agents. Its organizing abstraction is the **Kernel** — a small container into which you register two things: **AI services** (connectors to OpenAI, Azure OpenAI, Hugging Face, Ollama, …) and **plugins** (named groups of functions the model and your code can call). With those registered, you invoke functions, render prompt templates, and run agents through one consistent surface.

**The problem it solves.** Gluing an LLM into a production app means juggling a lot of plumbing: which model provider, how to format prompts, how to expose your existing code as tools the model can call, how to add logging/auth/telemetry around every model call, and how to swap providers without rewriting everything. SK standardizes that plumbing. Your business logic becomes **native functions** (decorated with `@kernel_function`); your prompts become **prompt functions** (template + model); the **Kernel** routes calls and the **connector** abstraction means switching from OpenAI to Azure is a one-line change.

**Reach for it when:** you're embedding AI into a larger application (especially a Microsoft/.NET shop, though Python and Java are first-class), you want one model-agnostic abstraction with enterprise concerns (filters for security/observability, structured telemetry) built in, or you want to expose existing code as tools for **automatic function calling**. **Skip it when** you just need a quick RAG chain (a thin [[langchain]] LCEL pipeline or raw SDK calls are simpler), or your problem is fundamentally a custom cyclic agent graph (use [[langgraph]]).

SK has grown beyond a function dispatcher: it now includes an **Agent Framework** (`ChatCompletionAgent`, group chats) and a **Process Framework** (stateful, event-driven business workflows). The original **Planners** that auto-assembled a plan from your functions are now largely superseded by **automatic function calling** (the model picks tools via native tool-calling).

## 2. Mental Model

Think of the Kernel as a **dependency-injection container with a function dispatcher** — the way an ASP.NET app has a service container, an SK app has a Kernel.

- **AI services (connectors)** are the *engines* you register: a chat-completion service, an embedding service. Swap the connector, keep the code.
- **Plugins** are *toolboxes*: a named bundle of **functions**. Functions come in two flavors — **native** (your Python/C# code, marked with `@kernel_function`) and **prompt** (a prompt template whose implementation *is* the LLM).
- **The Kernel** is the *switchboard*: you ask it to invoke `plugin.function(args)`, and it handles rendering, the model call, filters, and the return value.
- **Automatic function calling** is the kernel handing the model a catalog of your functions and letting *it* decide which to call — the modern replacement for hand-written Planners.

```
                       ┌──────────────── Kernel ────────────────┐
                       │  (DI container + dispatcher)            │
   your app  ──invoke──▶                                         │
                       │   AI services:  [ChatCompletion]        │
                       │                 [Embeddings]            │
                       │                                         │
                       │   plugins:                              │
                       │     math  → add(), square()   (native)  │
                       │     fun   → joke()            (prompt)  │
                       │                                         │
                       │   filters: [logging][auth][cache]       │
                       └─────────────────────────────────────────┘
                                       │
          direct invoke  ─────────────┤
          auto function calling ──────┘  (model reads the plugin catalog,
                                          picks a function, kernel runs it)
```

Key intuition: **everything the model can do is a registered function, and everything you register the model can be allowed to call.** Native code and prompts share one uniform interface, so a prompt function can call a native one and vice versa.

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **`Kernel`** | The central container/dispatcher. Holds services + plugins + filters. You call `kernel.invoke(func, **args)` or `kernel.invoke_prompt(...)`. |
| **AI service / connector** | A pluggable model backend registered with `kernel.add_service(...)`: `OpenAIChatCompletion`, `AzureChatCompletion`, `HuggingFace`, `Ollama`, plus embedding services. Provider-agnostic. |
| **Plugin** | A named collection of related functions, added via `kernel.add_plugin(instance, plugin_name=...)`. The unit of capability the model sees. |
| **Native function** | Ordinary code exposed as a function via the `@kernel_function` decorator. Type hints + description become the tool schema the model reads. |
| **Prompt function** | A function whose implementation is a **prompt template** + a model. Added via `kernel.add_function(prompt=..., ...)`. Templates use `{{$var}}` and can call other functions. |
| **`KernelArguments`** | The bag of variables *and* execution settings (temperature, model id, function-choice behavior) passed into an invocation. |
| **Automatic function calling** | `FunctionChoiceBehavior.Auto()` — the kernel exposes the plugin catalog to the model and executes whatever functions it chooses, looping until done. |
| **Planner** *(legacy)* | Older components that asked an LLM to assemble a multi-step plan from available functions. Mostly replaced by automatic function calling. |
| **Memory / vector store** | Embedding-backed semantic memory and connectors to vector DBs for RAG. |
| **Filters** | Cross-cutting hooks around invocations: function-invocation, prompt-render, and auto-function-invocation filters — for logging, auth, caching, PII redaction. |
| **Agent / Process frameworks** | Higher-level layers: `ChatCompletionAgent` + group chat for multi-agent; the Process Framework for stateful event-driven workflows. |

## 4. Setup

Semantic Kernel ships as one package per language. For Python:

```bash
pip install semantic-kernel              # core: Kernel, plugins, connectors
export OPENAI_API_KEY=...                # or AZURE_OPENAI_* for Azure
```

Requires **Python 3.10+**. The SDK is also available for **.NET** (`Microsoft.SemanticKernel`, the most mature target) and **Java**. There is no offline default model — like most LLM frameworks SK needs a configured connector before it can call anything.

Examples 1 & 2 below are **dependency-free**: they reimplement SK's core (`Kernel`, the `@kernel_function` decorator, native + prompt functions, automatic function calling) in ~40 lines of plain Python with a deterministic fake model, so the notebook runs anywhere with no install or API key. Example 3 shows the **real Semantic Kernel code** and executes it only if `semantic_kernel` is installed *and* `OPENAI_API_KEY` is set; otherwise it prints the exact code you'd write.

In [1]:
# Installs are optional — examples 1 & 2 are pure stdlib and run offline.
# Uncomment to get the real library used in example 3:
# %pip install semantic-kernel

import importlib.util, os

has_sk  = importlib.util.find_spec("semantic_kernel") is not None
has_key = bool(os.getenv("OPENAI_API_KEY"))
print("semantic_kernel installed:", has_sk)
print("OPENAI_API_KEY present:   ", has_key)

semantic_kernel installed: False
OPENAI_API_KEY present:    False


## 5. Worked Examples

### Example 1 — A Kernel with native plugins (the `@kernel_function` pattern)

SK's foundation is small: you decorate code with `@kernel_function`, group functions into **plugins**, register them on a **Kernel**, and invoke `plugin.function(args)`. The decorator attaches metadata (name + description) that later becomes the tool schema a model reads. Below we reimplement that core in plain Python so you can see exactly what `kernel.add_plugin(...)` and `kernel.invoke(...)` do under the hood — this mirrors the real `semantic_kernel.Kernel` and `@kernel_function`.

In [2]:
# --- a stand-in for semantic_kernel.functions.kernel_function ---
def kernel_function(description="", name=None):
    """Mark a method as callable by the kernel and attach tool metadata."""
    def deco(fn):
        fn.__kernel_function__ = True
        fn.__kernel_name__ = name or fn.__name__
        fn.__kernel_description__ = description
        return fn
    return deco


class Kernel:
    """Minimal stand-in for semantic_kernel.Kernel: a DI container + dispatcher."""
    def __init__(self):
        self.services = {}     # service_id -> AI connector
        self.plugins = {}      # plugin_name -> {func_name: callable}

    def add_service(self, service_id, service):
        self.services[service_id] = service

    def add_plugin(self, instance, plugin_name):
        funcs = {
            getattr(instance, attr).__kernel_name__: getattr(instance, attr)
            for attr in dir(instance)
            if getattr(getattr(instance, attr), "__kernel_function__", False)
        }
        self.plugins[plugin_name] = funcs
        return funcs

    def invoke(self, plugin_name, function_name, **kwargs):
        return self.plugins[plugin_name][function_name](**kwargs)


# --- plugins are just classes with decorated methods ---
class MathPlugin:
    @kernel_function(description="Add two integers")
    def add(self, a: int, b: int) -> int:
        return a + b

    @kernel_function(description="Square an integer")
    def square(self, x: int) -> int:
        return x * x


class TextPlugin:
    @kernel_function(description="Shout the text in upper case")
    def shout(self, text: str) -> str:
        return text.upper() + "!"


kernel = Kernel()
kernel.add_plugin(MathPlugin(), plugin_name="math")
kernel.add_plugin(TextPlugin(), plugin_name="text")

print("math.add(40, 2)      ->", kernel.invoke("math", "add", a=40, b=2))
print("text.shout('hi sk')  ->", kernel.invoke("text", "shout", text="hi sk"))

print("\nFunction catalog the model would see:")
for pname, funcs in kernel.plugins.items():
    for fname, fn in funcs.items():
        print(f"  {pname}.{fname:7} — {fn.__kernel_description__}")

math.add(40, 2)      -> 42
text.shout('hi sk')  -> HI SK!

Function catalog the model would see:
  math.add     — Add two integers
  math.square  — Square an integer
  text.shout   — Shout the text in upper case


### Example 2 — Prompt functions + automatic function calling

The other half of SK is **prompt functions** (a template whose implementation is the model) and **automatic function calling** (handing the model the catalog from Example 1 and letting *it* pick which function to run). Here a deterministic fake model stands in for the LLM: first we render an SK-style `{{$var}}` template, then we run a one-step function-calling loop where the "model" reads the goal, chooses a registered function, and the kernel executes it. This is the shape of `FunctionChoiceBehavior.Auto()`.

In [3]:
# --- a prompt (semantic) function: template + model as its implementation ---
def fake_chat(prompt: str) -> str:
    """Deterministic stand-in for a chat-completion connector."""
    return "It's like a super-fast helper that follows recipes called functions."


class PromptFunction:
    def __init__(self, template, service):
        self.template, self.service = template, service

    def render(self, **args):                       # SK uses {{$var}} syntax
        out = self.template
        for k, v in args.items():
            out = out.replace("{{$" + k + "}}", str(v))
        return out

    def invoke(self, **args):
        return self.service(self.render(**args))


explain = PromptFunction("Explain {{$topic}} to a five-year-old.", fake_chat)
print("prompt function ->", explain.invoke(topic="a kernel"))


# --- automatic function calling: the model picks a function from the catalog ---
def planning_model(goal: str, catalog: dict):
    """A real model emits a tool call; here we choose deterministically."""
    if "add" in goal or "plus" in goal:
        return ("math", "add", {"a": 40, "b": 2})
    if "square" in goal:
        return ("math", "square", {"x": 9})
    if "shout" in goal or "upper" in goal:
        return ("text", "shout", {"text": "automatic calling works"})
    return None


for goal in ["What is 40 plus 2?", "Please square nine.", "shout the result"]:
    choice = planning_model(goal, kernel.plugins)
    plugin, fn, fargs = choice
    result = kernel.invoke(plugin, fn, **fargs)     # kernel runs the chosen function
    print(f"goal={goal!r:28} -> model called {plugin}.{fn}{tuple(fargs.values())} = {result}")

prompt function -> It's like a super-fast helper that follows recipes called functions.
goal='What is 40 plus 2?'         -> model called math.add(40, 2) = 42
goal='Please square nine.'        -> model called math.square(9,) = 81
goal='shout the result'           -> model called text.shout('automatic calling works',) = AUTOMATIC CALLING WORKS!


### Example 3 — The real thing: a live Kernel with OpenAI

This is the production pattern with real SK classes. It calls a model, so it runs **only** if `semantic_kernel` is installed *and* `OPENAI_API_KEY` is set; otherwise it prints the exact code you'd write. Note the shape is identical to Examples 1–2 — register a service, register a plugin, add a prompt function, and let the model auto-invoke your native function — just executed by the real framework (SK's API is `async`).

In [4]:
SNIPPET = '''
import asyncio
from semantic_kernel import Kernel
from semantic_kernel.functions import kernel_function, KernelArguments
from semantic_kernel.connectors.ai.open_ai import (
    OpenAIChatCompletion, OpenAIChatPromptExecutionSettings,
)
from semantic_kernel.connectors.ai import FunctionChoiceBehavior


class MathPlugin:
    @kernel_function(description="Add two integers")
    def add(self, a: int, b: int) -> int:
        return a + b


async def main():
    kernel = Kernel()
    kernel.add_service(OpenAIChatCompletion(ai_model_id="gpt-4o-mini", service_id="chat"))
    kernel.add_plugin(MathPlugin(), plugin_name="math")

    # 1) a prompt (semantic) function from a template
    joke = kernel.add_function(
        plugin_name="fun", function_name="joke",
        prompt="Tell a one-line joke about {{$topic}}.",
    )
    print((await kernel.invoke(joke, topic="kernels")).value)

    # 2) automatic function calling: let the model invoke MathPlugin.add
    settings = OpenAIChatPromptExecutionSettings(service_id="chat")
    settings.function_choice_behavior = FunctionChoiceBehavior.Auto()
    answer = await kernel.invoke_prompt(
        "What is 40 plus 2? Use the available tools.",
        arguments=KernelArguments(settings=settings),
    )
    print(answer)

asyncio.run(main())
'''

import os, importlib.util
if importlib.util.find_spec("semantic_kernel") and os.getenv("OPENAI_API_KEY"):
    print("Running real Semantic Kernel...\n")
    exec(SNIPPET)
else:
    print("Skipping live run (need `pip install semantic-kernel` + OPENAI_API_KEY).")
    print("The code you would run:\n")
    print(SNIPPET)

Skipping live run (need `pip install semantic-kernel` + OPENAI_API_KEY).
The code you would run:


import asyncio
from semantic_kernel import Kernel
from semantic_kernel.functions import kernel_function, KernelArguments
from semantic_kernel.connectors.ai.open_ai import (
    OpenAIChatCompletion, OpenAIChatPromptExecutionSettings,
)
from semantic_kernel.connectors.ai import FunctionChoiceBehavior


class MathPlugin:
    @kernel_function(description="Add two integers")
    def add(self, a: int, b: int) -> int:
        return a + b


async def main():
    kernel = Kernel()
    kernel.add_service(OpenAIChatCompletion(ai_model_id="gpt-4o-mini", service_id="chat"))
    kernel.add_plugin(MathPlugin(), plugin_name="math")

    # 1) a prompt (semantic) function from a template
    joke = kernel.add_function(
        plugin_name="fun", function_name="joke",
        prompt="Tell a one-line joke about {{$topic}}.",
    )
    print((await kernel.invoke(joke, topic="kernels")).value)

    # 2) auto

## 6. Gotchas & Pitfalls

- **It's async-first in Python.** Most real calls (`kernel.invoke`, `invoke_prompt`) are coroutines — you `await` them inside `asyncio.run(...)`. Forgetting the `await` gives you a coroutine object, not a result.
- **Descriptions and type hints *are* the tool schema.** With automatic function calling, the model picks functions based on the `@kernel_function` description and the parameter type hints. Vague or missing descriptions → the model calls the wrong function or skips it. Treat them as the API contract, not a comment.
- **Planners are legacy.** Lots of older tutorials use `SequentialPlanner` / `StepwisePlanner`. The current idiom is `FunctionChoiceBehavior.Auto()` (native tool-calling). Don't build new work on the deprecated planners.
- **Fast-moving, multi-language API churn.** Namespaces and import paths (especially under `connectors.ai`) have shifted across versions, and .NET is usually ahead of Python/Java. Pin a version and check the docs for *that* release rather than trusting old blog posts.
- **No offline model.** Like every LLM framework, SK needs a configured connector + credentials. With no `OPENAI_API_KEY` / Azure config, the first model call fails — there's no built-in fallback.
- **`service_id` must match.** When you register multiple services, execution settings reference a service by `service_id`; a mismatch means the kernel can't find the model to use.
- **Auto function calling can loop and cost.** The model may chain several tool calls per request. Bound it (max iterations / function-choice config) and use **filters** to log and cap calls, or costs creep up silently.
- **Template syntax is literal.** Prompt templates use `{{$var}}` (and `{{plugin.func}}` to call functions). A typo'd variable name renders as the literal text rather than erroring — sanity-check rendered prompts when output looks off.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-offs vs Semantic Kernel |
|---|---|---|
| **Semantic Kernel** | Embedding AI into a larger app (esp. **.NET**/enterprise); model-agnostic connectors; mixing native code + prompts; built-in filters for security/observability | More ceremony than a quick script; multi-language API churn; lighter-weight on prebuilt agent recipes than CrewAI |
| **[[crewai]]** | Role-based multi-agent **pipelines** described in plain language; fast to stand up | Opinionated, Python-only, autonomy can wander; less of an app-framework / DI story |
| **[[langgraph]]** | Stateful, **cyclic** agent graphs; custom routing/branching; persistence + human-in-the-loop | Lower-level graph wiring; not a DI container or enterprise app framework |
| **[[autogen]]** | Free-form **conversational** multi-agent systems with code execution | Conversation-centric; also Microsoft, with overlapping goals (the two are converging) |
| **[[langchain]] (LCEL)** | **Linear** dataflow chains, RAG; huge integration catalog | Not built around a kernel/DI model; SK leans more enterprise/.NET and observability |
| **Raw provider SDK** | A single tool-using agent or one-off prompt; minimal dependencies | You hand-roll function-calling, templating, provider-switching, and filters that SK gives you |

**Rule of thumb:** if the LLM is *one component inside a real application* — particularly a Microsoft/.NET stack where you want provider-agnostic connectors, your existing code exposed as tools, and filters for auth/logging — Semantic Kernel is the natural fit. If you want a batteries-included multi-agent *pipeline*, reach for CrewAI; if you need a custom stateful loop, reach for LangGraph.

> **Note:** Microsoft's **AutoGen** and Semantic Kernel are actively converging toward a shared agent runtime, so expect the lines between them to blur.

## 8. Resources

- **Official docs** — https://learn.microsoft.com/en-us/semantic-kernel/overview/
- **Python getting started** — https://learn.microsoft.com/en-us/semantic-kernel/get-started/quick-start-guide
- **Plugins & functions** — https://learn.microsoft.com/en-us/semantic-kernel/concepts/plugins/
- **Function calling (auto tool use)** — https://learn.microsoft.com/en-us/semantic-kernel/concepts/ai-services/chat-completion/function-calling/
- **Agent Framework** — https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/
- **GitHub** — https://github.com/microsoft/semantic-kernel